In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

In [2]:
df = pd.read_csv('homework_2.2.csv')
df

,X,Y,Z
0,0,1.182435,-0.725820
1,0,2.714474,0.563476
2,0,0.077612,-0.435632
3,0,-0.154449,-0.104553
4,0,22.298992,-2.321273
...,...,...,...
9995,0,0.019371,-0.409462
9996,0,2.581533,0.545860
9997,0,0.209599,-0.486216
9998,0,16.829356,-2.045500


For questions 3-5:

Given a data set, create a bootstrap simulation to try different possibilities. 
Use the file homework_2.2.csv 

Q3: If we were to measure the effect of the treatment simply by subtracting the outcomes of the treated vs. untreated population, which of these is closest to the mean effect? (This is not the recommended way of measuring the mean effect when there are confounders!) 



In [ ]:
means = df.groupby('X')[['Y', 'Z']].mean()

# diff in untreated/treated means
eff = means.loc[1] - means.loc[0]
eff

Y    2.920717
Z    0.048016
dtype: float64

Q4: If we were to use bootstrap sampling to measure the variance of that effect, again finding the effect using the non-recommended approach, which of these is closest to that variance?

In [ ]:
import numpy as np
import pandas as pd

def get_treatment_effect(data):
    # diff in untreated/treated means
    means = data.groupby("X")[["Y", "Z"]].mean()
    return means.loc[1] - means.loc[0]

bootstrap_sim_trial_count = 10000
bootstrap_effects = []

for _ in range(bootstrap_sim_trial_count):
    boot_sample = df.sample(n=len(df), replace=True) # sample with replacement
    effect = get_treatment_effect(boot_sample)
    bootstrap_effects.append(effect)

bootstrap_df = pd.DataFrame(bootstrap_effects)
bootstrap_variances = bootstrap_df.var()
bootstrap_variances

Y    0.032555
Z    0.000610
dtype: float64

Q5: If we ran a linear regression (with intercept) to measure the effect, which of these is closest to the skewness of the effect measured? (Look up skewness online. You can use scipy.stats.skew to compute the skewness of a list of numbers.)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skew
import statsmodels.api as sm

def get_regression_effect(data):
    # Add a constant for the intercept
    X_reg = sm.add_constant(data["X"])
    model = sm.OLS(data["Y"], X_reg).fit()
    # Return the coefficient of the treatment variable X
    return model.params["X"]

bootstrap_sim_trial_count = 10_000
bootstrap_coefs = []

for _ in range(bootstrap_sim_trial_count):
    boot_sample = df.sample(n=len(df), replace=True)
    coef = get_regression_effect(boot_sample)
    bootstrap_coefs.append(coef)

effect_skewness = skew(bootstrap_coefs)
effect_skewness

Skewness of the treatment effect: 0.08210989734045372


In [ ]:
# Reflection 2 question 2

# Create pareto dist
a, m = 3., 2.  # pareto shape and mode (default values from np docs)
# sample_size = 1_000 # Variance of the sample mean: 0.002300
# sample_size = 5_000 # Variance of the sample mean: 0.000441
# sample_size = 10_000 # Variance of the sample mean: 0.000229
sample_size = 100_000 # Variance = 0.000028
s = (np.random.pareto(a, sample_size) + 1) * m

# plt.hist(s, bins=30)

bootstrap_sim_trial_count = 1000  # Number of bootstrap samples to make to be used for mean/var calculation
bootstrap_means = []

for _ in range(bootstrap_sim_trial_count):
    boot_sample = np.random.choice(s, size=sample_size, replace=True) # Sample with replacement
    boot_mean = np.mean(boot_sample)
    bootstrap_means.append(boot_mean)

variance_of_means = np.var(bootstrap_means)
print(f"{variance_of_means:.8f}")

0.00002486
